# ci-triage: Colab setup

Run this at the start of every Colab session. It clones the repo, installs the package and runs the test suite.

The code lives in `src/ci_triage/`; notebooks only call it. Edit files in VS Code, `git push`, then re-run the clone cell here to pick up changes.

In [ ]:
REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/llm-ci-triage.git"  # edit me
BRANCH = "main"
REPO_DIR = "/content/llm-ci-triage"

In [ ]:
# Fresh clone each time so the notebook always runs what's on GitHub.
!rm -rf {REPO_DIR}
!git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}
!pip install --quiet -e "{REPO_DIR}[dev]"

In [ ]:
# An editable install is only picked up after a runtime restart, so add src/ to the path directly.
import sys

sys.path.insert(0, f"{REPO_DIR}/src")

import ci_triage

print("ci_triage", ci_triage.__version__)

In [ ]:
!cd {REPO_DIR} && ruff check . && mypy src tests && pytest

## Try it on the sample event

Parses the test fixture and renders the summary the Action writes, without calling GitHub.

In [ ]:
import json
from pathlib import Path

from IPython.display import Markdown

from ci_triage.event import load_event
from ci_triage.github import Job
from ci_triage.summary import render_summary

fixtures = Path(REPO_DIR) / "tests" / "fixtures"
event = load_event(fixtures / "workflow_run_failure.json")
jobs = [Job.model_validate(j) for j in json.loads((fixtures / "jobs.json").read_text())["jobs"]]
failed = [j for j in jobs if j.conclusion == "failure"]

Markdown(render_summary(event, failed))

## GPU check (used from week 4)

Runtime → Change runtime type → T4 GPU. Not needed yet.

In [ ]:
!nvidia-smi || echo "No GPU attached (fine for week 1)."